# Tutorial 02 — Drive the counter via the eda-agents Python API

> **EXPERIMENTAL.** This tutorial uses the [`eda-agents`](https://github.com/Mauricio-xx/eda-agents) Python API to drive LibreLane on your behalf. Not part of the chipathon tapeout signoff path — for tapeout work stay with `examples/`.

Where Tutorial 01 had you chat with the agent in a terminal, this tutorial does the same flow programmatically: ~30 lines of Python that wrap the LibreLane config in a `GenericDesign` and hand it to a `ProjectManager` (multi-agent hierarchy: SynthesisEngineer + PhysicalDesigner + VerificationEngineer + SignoffChecker). No chat session. Useful when you want to script the agent into a larger pipeline (CI, bench harness, cron job).

**Pre-requisites** (one-time, see [`../docs/eda_agents_setup.md`](../docs/eda_agents_setup.md)):

- `gf180` container running.
- `eda-agents` pip-installed in an active venv with the `[adk]` extra.
- One of: `claude` CLI on PATH (`BACKEND="cc_cli"`, free with subscription) **or** `OPENROUTER_API_KEY` (`BACKEND="adk"`).


## Step 0 — Configuration + path guard

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

# --- Toggles --------------------------------------------------------
RUN_PIP_INSTALL    = False  # pip install eda-agents (only needed once per venv)
RUN_PREFLIGHT      = True   # check container, eda-agents, claude
RUN_STAGE          = True   # copy rtl/ tb/ librelane/ to ~/eda/designs/
RUN_DRY_PM         = True   # construct ProjectManager + dry-run, ~5s, free
RUN_REAL           = False  # full LibreLane flow via the agent (~10-15 min)
# cc_cli backend launches `claude --print`; tool calls (docker, file)
# need --dangerously-skip-permissions in non-interactive mode. Double-
# gated: requires both this flag and `EDA_AGENTS_ALLOW_DANGEROUS=1` env.
RUN_DANGEROUSLY    = False  # only flip with cc_cli + RUN_REAL
RUN_METRICS_CHECK  = False  # parse final/metrics.csv after RUN_REAL

# --- Backend choice -------------------------------------------------
BACKEND   = 'cc_cli'   # 'cc_cli' (Claude subscription) | 'adk' (API key)
# Backend-aware default model. cc_cli routes to `claude --print`, which only accepts
# Anthropic model IDs (claude-sonnet-4-6 etc.) -- passing a Google or OpenRouter ID
# returns API 404. adk and other backends route through LiteLLM and accept
# provider-prefixed IDs like 'google/gemini-3-flash-preview'.
_DEFAULT_MODEL = 'claude-sonnet-4-6' if BACKEND == 'cc_cli' else 'google/gemini-3-flash-preview'
LLM_MODEL = os.environ.get('EDA_AGENTS_MODEL', _DEFAULT_MODEL)
MAX_BUDGET_USD = 1.00

# --- Path guard -----------------------------------------------------
try:
    NB_DIR = Path(__file__).resolve().parent
except NameError:
    NB_DIR = Path.cwd().resolve()

for _sub in ('rtl', 'tb', 'librelane'):
    if not (NB_DIR / _sub).is_dir():
        raise RuntimeError(
            f'NB_DIR={NB_DIR} does not contain {_sub}/. '
            'Launch from tutorials/02_counter_python_api/.'
        )

REPO_ROOT      = NB_DIR.parents[1]
HOST_WORKSPACE = Path.home() / 'eda' / 'designs' / 'eda_agents_counter_pyapi'
WORK_DIR       = NB_DIR / 'rtl2gds_counter_pyapi_results'
EDA_AGENTS_ROOT = Path(
    os.environ.get('EDA_AGENTS_ROOT', Path.home() / 'personal_exp' / 'eda-agents')
).resolve()

print(f'NB_DIR          : {NB_DIR}')
print(f'EDA_AGENTS_ROOT : {EDA_AGENTS_ROOT}  (exists: {EDA_AGENTS_ROOT.is_dir()})')
print(f'HOST_WORKSPACE  : {HOST_WORKSPACE}')
print(f'WORK_DIR        : {WORK_DIR}')
print(f'BACKEND         : {BACKEND}')


## Step 1 — (Optional) install eda-agents into the active venv

Only needed once per venv. If `import eda_agents` already works, leave `RUN_PIP_INSTALL=False`.

In [ ]:
import sys

if RUN_PIP_INSTALL:
    if not EDA_AGENTS_ROOT.is_dir():
        raise RuntimeError(
            f'EDA_AGENTS_ROOT does not exist: {EDA_AGENTS_ROOT}\n'
            'Either clone https://github.com/Mauricio-xx/eda-agents into '
            f'{EDA_AGENTS_ROOT.parent} or set $EDA_AGENTS_ROOT.'
        )
    if 'VIRTUAL_ENV' not in os.environ:
        print('WARNING: no $VIRTUAL_ENV. You probably want to be in a venv.')
    cmd = [sys.executable, '-m', 'pip', 'install', '-e', f'{EDA_AGENTS_ROOT}[adk]']
    print('$ ' + ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('(skipped -- flip RUN_PIP_INSTALL=True if `import eda_agents` fails)')

## Step 2 — Pre-flight checks

In [ ]:
if RUN_PREFLIGHT:
    out = subprocess.run(
        ['docker', 'ps', '--filter', 'name=gf180', '--format', '{{.Names}}'],
        capture_output=True, text=True, timeout=10,
    )
    container_ok = 'gf180' in out.stdout.split()

    try:
        import eda_agents  # noqa: F401
        eda_ok = True
    except ImportError:
        eda_ok = False

    print(f'  gf180 container running : {container_ok}')
    print(f'  eda_agents importable   : {eda_ok}')
    print(f'  claude CLI on PATH      : {shutil.which("claude") or "-- not found"}')
    print(f'  OPENROUTER_API_KEY set  : {bool(os.environ.get("OPENROUTER_API_KEY"))}')
    print(f'  GOOGLE_API_KEY set      : {bool(os.environ.get("GOOGLE_API_KEY"))}')

    if not container_ok:
        print('FIX: scripts/bootstrap_container.sh from repo root.')
    if not eda_ok:
        print('FIX: flip RUN_PIP_INSTALL=True in step 1, or pip install eda-agents manually.')
    if BACKEND == 'cc_cli' and not shutil.which('claude'):
        print('FIX: install Claude Code or switch BACKEND="adk" + set OPENROUTER_API_KEY.')
else:


## Step 3 — Stage workspace

Same as Tutorial 01 but to a different host path so the two flows do not collide.

In [ ]:
if RUN_STAGE:
    HOST_WORKSPACE.mkdir(parents=True, exist_ok=True)
    for sub in ('rtl', 'tb', 'librelane'):
        src = NB_DIR / sub
        dst = HOST_WORKSPACE / sub
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'  staged {src.relative_to(REPO_ROOT)} -> {dst}')
    print(f'\nWorkspace ready: {HOST_WORKSPACE}')
else:
    print('(skipped -- flip RUN_STAGE=True)')

## Step 4 — Construct the agent + dry-run

Free, ~5 seconds. Wraps the LibreLane config with `GenericDesign` (auto-derives the 13 `DigitalDesign` methods from the YAML) and hands it to `ProjectManager` (master ADK / cc_cli agent that orchestrates 4 specialists).

Dry-run prints the prompt length (cc_cli) or sub-agent topology (adk) without making any LLM call.

In [ ]:
import asyncio

if RUN_DRY_PM:
    from eda_agents.core.designs.generic import GenericDesign
    from eda_agents.agents.digital_adk_agents import ProjectManager

    pdk_root = os.environ.get('PDK_ROOT') or None
    design = GenericDesign(
        config_path=str(HOST_WORKSPACE / 'librelane' / 'config.yaml'),
        pdk_root=pdk_root,
        pdk_config='gf180mcu',
    )
    pm = ProjectManager(
        design=design,
        model=LLM_MODEL,
        backend=BACKEND,
        max_budget_usd=MAX_BUDGET_USD,
    )

    async def _dry():
        return await pm.run(WORK_DIR, dry_run=True)

    result = asyncio.run(_dry())

    print(f'design          : {design.project_name()}')
    print(f'specs           : {design.specs_description()}')
    print(f'FoM             : {design.fom_description()}')
    print(f'backend         : {BACKEND}')
    print(f'model           : {LLM_MODEL}')
    if BACKEND == 'cc_cli':
        print(f'prompt length   : {result.get("prompt_length", 0)} chars')
    else:
        subs = result.get('sub_agent_names') or result.get('sub_agents') or []
        print(f'master          : {result.get("master_agent", "N/A")}')
        print(f'sub-agents      : {", ".join(str(s) for s in subs)}')
else:
    print('(skipped -- flip RUN_DRY_PM=True for the dry-run)')

## Step 5 — Real run

**~10-15 minutes.** Hands the design to the agent, which calls LibreLane via Docker, parses the metrics, and reports back. Watch the `WORK_DIR` for the agent's log.

In [ ]:
import json

if RUN_REAL:
    from eda_agents.core.designs.generic import GenericDesign
    from eda_agents.agents.digital_adk_agents import ProjectManager

    pdk_root = os.environ.get('PDK_ROOT') or None
    design = GenericDesign(
        config_path=str(HOST_WORKSPACE / 'librelane' / 'config.yaml'),
        pdk_root=pdk_root,
        pdk_config='gf180mcu',
    )
    if BACKEND == 'cc_cli' and not (
        RUN_DANGEROUSLY and os.environ.get('EDA_AGENTS_ALLOW_DANGEROUS') == '1'
    ):
        raise RuntimeError(
            'cc_cli backend in non-interactive mode needs --dangerously-skip-permissions. '
            'Flip RUN_DANGEROUSLY=True in cell 0 AND export EDA_AGENTS_ALLOW_DANGEROUS=1 '
            'before launching this cell.'
        )
    pm = ProjectManager(
        design=design,
        model=LLM_MODEL,
        backend=BACKEND,
        max_budget_usd=MAX_BUDGET_USD,
        allow_dangerous=RUN_DANGEROUSLY,
    )

    async def _real():
        return await pm.run(WORK_DIR)

    result = asyncio.run(_real())
    safe = {k: v for k, v in result.items() if k not in ('agent_output', 'prompt')}
    print(json.dumps(safe, indent=2, default=str))
else:
    print('(skipped -- flip RUN_REAL=True after RUN_DRY_PM completes cleanly)')


## Step 6 — Verify signoff metrics from the host side

In [ ]:
import csv

SIGNOFF_KEYS = [
    'design__instance__count__stdcell',
    'timing__setup_vio__count',
    'timing__hold_vio__count',
    'magic__drc_error__count',
    'klayout__drc_error__count',
    'design__lvs_error__count',
    'route__drc_errors',
    'power__total',
]

if RUN_METRICS_CHECK:
    # LibreLane writes runs/ next to config.yaml, which here is
    # HOST_WORKSPACE/librelane/.
    runs_dir = HOST_WORKSPACE / 'librelane' / 'runs'
    if not runs_dir.exists():
        print(f'No runs/ directory at {runs_dir}.')
        print('Did step 5 succeed?')
    else:
        latest_run = sorted(runs_dir.iterdir(), key=lambda p: p.stat().st_mtime)[-1]
        metrics_path = latest_run / 'final' / 'metrics.csv'
        if not metrics_path.exists():
            print(f'metrics.csv missing under {latest_run}.')
        else:
            print(f'Reading {metrics_path}\n')
            found = {}
            with metrics_path.open() as fh:
                for row in csv.reader(fh):
                    if row and row[0] in SIGNOFF_KEYS:
                        found[row[0]] = row[1] if len(row) > 1 else ''
            for key in SIGNOFF_KEYS:
                print(f'  {key:45s} {found.get(key, "(missing)")}')
            print()
            zero_keys = [
                'timing__setup_vio__count',
                'timing__hold_vio__count',
                'magic__drc_error__count',
                'design__lvs_error__count',
                'route__drc_errors',
            ]
            offenders = [k for k in zero_keys if found.get(k, '0') not in ('0', '')]
            if offenders:
                print(f'FAIL: non-zero signoff metric(s): {offenders}')
            else:
                print('PASS: all hard-zero signoff metrics are 0.')
else:
    print('(skipped -- flip RUN_METRICS_CHECK=True after the agent reports success)')


## What you learned

- `GenericDesign(config.yaml)` auto-derives the 13 `DigitalDesign` methods from a LibreLane config — zero subclassing needed.
- `ProjectManager` is the master agent; it orchestrates `SynthesisEngineer`, `PhysicalDesigner`, `VerificationEngineer`, `SignoffChecker`.
- `backend="cc_cli"` runs the master as a Claude Code subprocess (free with subscription); `backend="adk"` runs it as a Google ADK LlmAgent (needs an API key).
- `pm.run(work_dir, dry_run=True)` is free and shows the prompt + sub-agent topology before you spend money / wall time.
- The `run_counter.py` twin in this directory does the same thing as a single Python script.

## Next

[`../03_counter_autoresearch/`](../03_counter_autoresearch/) closes the loop: instead of one agent run, a greedy autoresearch loop sweeps QoR knobs (density, clock period, PDN width) under a $0.30 budget cap.